## EJERCICIO 2

In [25]:
import numpy as np
import random
import copy
import csv
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# INDIVIDUO BINARIO
class IndividuoBinario:
    def __init__(self, longitud_cromosoma, prob_inicial_unos=0.1):
        self.genotipo = np.random.choice([0, 1], size=longitud_cromosoma, p=[1-prob_inicial_unos, prob_inicial_unos])
        self.fitness = None
        self.longitud = longitud_cromosoma

    def mutar(self, tasa_mutacion):
        for i in range(self.longitud):
            if random.random() < tasa_mutacion:
                self.genotipo[i] = 1 - self.genotipo[i]
                break

    def cruzar(self, madre):
        hijo1 = copy.deepcopy(self)
        hijo2 = copy.deepcopy(madre)
        
        punto_cruza = random.randint(1, self.longitud - 2)
        hijo1.genotipo = np.concatenate((self.genotipo[:punto_cruza], madre.genotipo[punto_cruza:]))
        hijo2.genotipo = np.concatenate((madre.genotipo[:punto_cruza], self.genotipo[punto_cruza:]))
        
        return hijo1, hijo2


# GENETICO BINARIO
class AlgoritmoGeneticoBinario:
    def __init__(self, X_train, y_train, penalizacion=0.2, tam_poblacion=20, generaciones=30, tasa_mutacion=0.01):
        self.X_train = X_train
        self.y_train = y_train
        self.penalizacion = penalizacion
        self.longitud = X_train.shape[1]
        self.tam_poblacion = tam_poblacion
        self.generaciones = generaciones
        self.tasa_mutacion = tasa_mutacion
        
        self.poblacion = []
        self.mejor_individuo = None

    def evaluar(self, individuo):
        indices_seleccionados = np.where(individuo.genotipo == 1)[0]
        num_seleccionadas = len(indices_seleccionados)
        
        if num_seleccionadas == 0:
            individuo.fitness = -1.0 
        else:
            X_train_sel = self.X_train[:, indices_seleccionados]
            
            clf = KNeighborsClassifier(n_neighbors=3)
            clf.fit(X_train_sel, self.y_train)
            
            acc = accuracy_score(self.y_train, clf.predict(X_train_sel))
            
            porcion_usada = num_seleccionadas / self.longitud
            individuo.fitness = acc - (self.penalizacion * porcion_usada)

        if self.mejor_individuo is None or individuo.fitness > self.mejor_individuo.fitness:
            self.mejor_individuo = copy.deepcopy(individuo)

    def inicializar_poblacion(self):
        self.poblacion = [IndividuoBinario(self.longitud) for _ in range(self.tam_poblacion)]
        for ind in self.poblacion:
            self.evaluar(ind)

    def seleccion_comp(self, k=3):
        competencia = random.sample(self.poblacion, k)
        def obtener_fitness(ind):
            return ind.fitness
        return max(competencia, key=obtener_fitness)

    def evolucionar(self):
        nueva_poblacion = []
        
        nueva_poblacion.append(copy.deepcopy(self.mejor_individuo))

        while len(nueva_poblacion) < self.tam_poblacion:
            padre = self.seleccion_comp()
            madre = self.seleccion_comp()
            
            hijo1, hijo2 = padre.cruzar(madre)
            hijo1.mutar(self.tasa_mutacion)
            hijo2.mutar(self.tasa_mutacion)
            
            self.evaluar(hijo1)
            self.evaluar(hijo2)
            
            nueva_poblacion.append(hijo1)
            nueva_poblacion.append(hijo2)
            
        self.poblacion = nueva_poblacion[:self.tam_poblacion]

    def ejecutar(self):
        self.inicializar_poblacion()
        for gen in range(self.generaciones):
            self.evolucionar()
            print(f"Generación {gen+1}: Mejor Fitness = {self.mejor_individuo.fitness:.4f} | Genes usados: {np.sum(self.mejor_individuo.genotipo)}")
        return self.mejor_individuo


def cargar_datos(ruta_archivo):
    datos = np.loadtxt(ruta_archivo, delimiter=',')
    datos_x = datos[:, :-1] 
    yd = datos[:, -1].astype(int) 
    return datos_x, yd


if __name__ == "__main__":
    x_train, y_train = cargar_datos("datos/leukemia_train.csv")
    x_test, y_test = cargar_datos("datos/leukemia_test.csv")

    print("Iniciando evolucion: ")
    ag = AlgoritmoGeneticoBinario(x_train, y_train, penalizacion=0.5, tam_poblacion=30, generaciones=20, tasa_mutacion=0.01)
    mejor_solucion = ag.ejecutar()
    
    indices_finales = np.where(mejor_solucion.genotipo == 1)[0]

    print("\n -- RESULTADO FINAL ")
    print(f"Características seleccionadas: {len(indices_finales)} de {x_train.shape[1]}")
    
    # PRUEBA USANDO EL MISMO CLASIFICADOR (KNN)
    clf_final = KNeighborsClassifier(n_neighbors=3)
    clf_final.fit(x_train[:, indices_finales], y_train)
    predicciones_test = clf_final.predict(x_test[:, indices_finales])
    precision_test = accuracy_score(y_test, predicciones_test)
    
    print(f"Precisión en conjunto de prueba: {precision_test * 100:.2f}%")

Iniciando evolucion: 
Generación 1: Mejor Fitness = 0.9502 | Genes usados: 710
Generación 2: Mejor Fitness = 0.9502 | Genes usados: 710
Generación 3: Mejor Fitness = 0.9504 | Genes usados: 707
Generación 4: Mejor Fitness = 0.9504 | Genes usados: 707
Generación 5: Mejor Fitness = 0.9512 | Genes usados: 696
Generación 6: Mejor Fitness = 0.9512 | Genes usados: 696
Generación 7: Mejor Fitness = 0.9512 | Genes usados: 696
Generación 8: Mejor Fitness = 0.9512 | Genes usados: 696
Generación 9: Mejor Fitness = 0.9512 | Genes usados: 696
Generación 10: Mejor Fitness = 0.9513 | Genes usados: 695
Generación 11: Mejor Fitness = 0.9513 | Genes usados: 694
Generación 12: Mejor Fitness = 0.9515 | Genes usados: 691
Generación 13: Mejor Fitness = 0.9515 | Genes usados: 691
Generación 14: Mejor Fitness = 0.9516 | Genes usados: 690
Generación 15: Mejor Fitness = 0.9516 | Genes usados: 690
Generación 16: Mejor Fitness = 0.9516 | Genes usados: 690
Generación 17: Mejor Fitness = 0.9516 | Genes usados: 690
G